<a href="https://colab.research.google.com/github/balajiduddukuri/Langchain_practice/blob/Ultimate-Content-Repurposer/LangChain_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
pip install -q langchain langchain-google-genai langgraph langchain-experimental

In [55]:
import os
from google.colab import userdata

# LangChain core + Google Gemini Chat wrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from google.colab import userdata
userdata.get('GEMINI_API_KEY')
# Load and assert API key (store it in Colab with key name: "google_api_key")
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
assert os.environ.get("GOOGLE_API_KEY"), (
    "Missing GOOGLE_API_KEY. In Colab, run: "
    "from google.colab import userdata; userdata.set('google_api_key', 'YOUR_KEY')"
)
print("Google API key is loaded successfully.")

Google API key is loaded successfully.


In [24]:
import os
from typing import TypedDict, Annotated, List, Literal

from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable
from langchain_core.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.tools.python.tool import PythonREPLTool

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

In [69]:
MODEL_ID = "gemini-2.5-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL_ID,
    temperature=0.0,
)

In [70]:
# ---------------- STATE ----------------
class SDLCState(TypedDict):
    messages: Annotated[List[BaseMessage], "add_messages"]
    current_persona: str
    final_output: str




In [71]:

# ---------------- PERSONA PROMPTS ----------------
BA_SYSTEM = """
You are a Business Analyst persona.
Responsibilities:
- Clarify the requirements.
- Produce epics, user stories, and acceptance criteria.
When ready to hand off, end your response with a line:
NEXT: Product Manager
If you believe the whole workflow is already done, end with:
FINAL:
"""

PM_SYSTEM = """
You are a Product Manager persona.
Responsibilities:
- Prioritize scope.
- Define release plan and MVP.
When ready to hand off to implementation, end with:
NEXT: Developer
If everything is already complete, end with:
FINAL:
"""

DEV_SYSTEM = """
You are a Developer persona.
Responsibilities:
- Propose high-level design.
- Provide implementation details or code.
- You may use the `python_repl` tool to execute Python code when it helps:
  - Use it to quickly prototype or validate logic.
  - Include only minimal, relevant code in tool calls.

When you want to execute code, CALL the `python_repl` tool with the code string.
When ready for testing, end with:
NEXT: Tester
If everything is already complete, end with:
FINAL:
"""

QA_SYSTEM = """
You are a Tester persona.
Responsibilities:
- Design test strategy and test cases.
- Think about edge cases and non‑functional tests.
When ready to hand off to DevOps, end with:
NEXT: DevOps
If everything is already complete, end with:
FINAL:
"""

DEVOPS_SYSTEM = """
You are a DevOps Engineer persona.
Responsibilities:
- Plan CI/CD, deployment, monitoring, and rollback.
- Think about reliability, security, and observability.
When you are done, summarize the end‑to‑end SDLC plan.
Always end your message with:
FINAL:
"""

In [72]:
# ---------------- PYTHON REPL TOOL (Developer only) ----------------
_repl = PythonREPLTool()
@tool
def python_repl(code: str) -> str:
    """Execute Python code. Use print(...) to show values."""
    try:
        result = _repl.invoke(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return (
        "Successfully executed:\n```python\n"
        + code
        + "\n```\nStdout:\n"
        + str(result)
        + "\n\nIf you have completed all tasks, respond with FINAL:"
    )

dev_tools = [python_repl]

In [60]:
dev_tools = [python_repl]

In [73]:
# ---------------- PERSONA NODES ----------------
def make_persona_node(system_prompt: str, persona_name: str):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("placeholder", "{history}"),
            ("human", "{task}"),
        ]
    )
    chain: Runnable = prompt | llm

    def node(state: SDLCState) -> SDLCState:
        history = state.get("messages", [])
        last_human = None
        for m in reversed(history):
            if isinstance(m, HumanMessage):
                last_human = m
                break
        task_text = last_human.content if last_human else "Continue the SDLC workflow."
        ai_msg = chain.invoke({"history": history, "task": task_text})
        state["messages"] = history + [ai_msg]
        state["current_persona"] = persona_name
        return state

    return node


In [74]:
def make_persona_node_with_tools(system_prompt: str, persona_name: str, tools=None):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("placeholder", "{history}"),
            ("human", "{task}"),
        ]
    )
    model = llm
    if tools:
        model = llm.bind_tools(tools)  # Gemini 3 supports tool calling in preview. [web:66][web:69]
    chain: Runnable = prompt | model

    def node(state: SDLCState) -> SDLCState:
        history = state.get("messages", [])
        last_human = None
        for m in reversed(history):
            if isinstance(m, HumanMessage):
                last_human = m
                break
        task_text = last_human.content if last_human else "Continue the SDLC workflow."
        ai_msg = chain.invoke({"history": history, "task": task_text})
        state["messages"] = history + [ai_msg]
        state["current_persona"] = persona_name
        return state

    return node





In [75]:
ba_node = make_persona_node(BA_SYSTEM, "BA")
pm_node = make_persona_node(PM_SYSTEM, "PM")
dev_node = make_persona_node_with_tools(DEV_SYSTEM, "DEV", tools=dev_tools)
qa_node = make_persona_node(QA_SYSTEM, "QA")
devops_node = make_persona_node(DEVOPS_SYSTEM, "DEVOPS")

In [76]:
# ---------------- ROUTING ----------------
def route_next(state: SDLCState) -> Literal["BA", "PM", "DEV", "QA", "DEVOPS", "__end__"]:
    msgs = state.get("messages", [])
    if not msgs:
        return "BA"
    last = msgs[-1]
    content = last.content if isinstance(last, (AIMessage, HumanMessage)) else str(last)

    if "FINAL:" in content:
        state["final_output"] = content
        return "__end__"

    if "NEXT: Product Manager" in content:
        return "PM"
    if "NEXT: Developer" in content:
        return "DEV"
    if "NEXT: Tester" in content:
        return "QA"
    if "NEXT: DevOps" in content or "NEXT: DevOps Engineer" in content:
        return "DEVOPS"

    persona = state.get("current_persona", "BA")
    order = ["BA", "PM", "DEV", "QA", "DEVOPS"]
    try:
        idx = order.index(persona)
        if idx + 1 < len(order):
            return order[idx + 1]
        return "__end__"
    except ValueError:
        return "BA"


def persona_or_tools(state: SDLCState) -> Literal["persona", "tools"]:
    msgs = state.get("messages", [])
    if not msgs:
        return "persona"
    last = msgs[-1]
    if hasattr(last, "tool_calls") and getattr(last, "tool_calls"):
        return "tools"
    return "persona"

# ---------------- TOOL NODE ----------------
tool_node = ToolNode(dev_tools)

In [65]:
# ---------------- BUILD GRAPH ----------------
def passthrough_node(state: SDLCState) -> SDLCState:
    return state

def build_graph():
    workflow = StateGraph(SDLCState)

    workflow.add_node("BA", ba_node)
    workflow.add_node("PM", pm_node)
    workflow.add_node("DEV", dev_node)
    workflow.add_node("QA", qa_node)
    workflow.add_node("DEVOPS", devops_node)
    workflow.add_node("tools", tool_node)
    workflow.add_node("__route_next__", passthrough_node) # Add __route_next__ as a passthrough node

    workflow.add_edge(START, "BA")

    for name in ["BA", "PM", "DEV", "QA", "DEVOPS"]:
        workflow.add_conditional_edges(
            name,
            persona_or_tools,
            {
                "persona": "__route_next__",
                "tools": "tools",
            },
        )

    workflow.add_edge("tools", "DEV")

    # This block is now essential for routing between personas
    workflow.add_conditional_edges(
        "__route_next__",
        route_next,
        {
            "BA": "BA",
            "PM": "PM",
            "DEV": "DEV",
            "QA": "QA",
            "DEVOPS": "DEVOPS",
            "__end__": END,
        },
    )

    return workflow.compile()


In [66]:
graph = build_graph()

In [67]:
# ---------------- MAIN DEMO ----------------
def run_demo():
    user_task = (
        "Design and deliver a python program for finding string length "
    )

    initial_state: SDLCState = {
        "messages": [HumanMessage(content=user_task)],
        "current_persona": "BA",
        "final_output": "",
    }

    print(f"Using model: {MODEL_ID}")
    print("=== STREAMING STEPS ===")
    for step in graph.stream(initial_state):
        for node_name, state in step.items():
            print(f"\n--- Node: {node_name} (Persona: {state.get('current_persona')}) ---")
            last = state["messages"][-1]
            print(last.content[:1000])

    final_state = graph.invoke(initial_state)
    print("\n=== FINAL OUTPUT ===")
    print(final_state.get("final_output", ""))

In [77]:
run_demo()

Using model: gemini-2.5-flash
=== STREAMING STEPS ===


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}